# import libs

In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import joblib
import json
import os
from scipy import stats

# load model and params

In [28]:
# Load artifacts 
best_model = joblib.load("D:/credit-card-crosssell/models/xgb_propensity.pkl")
encoders   = joblib.load("D:/credit-card-crosssell/models/encoders.pkl")

with open("D:/credit-card-crosssell/models/selected_features.json") as f:
    selected = json.load(f)

with open("D:/credit-card-crosssell/models/best_params.json") as f:
    best_params = json.load(f)

with open("D:/credit-card-crosssell/models/model_results.json") as f:
    results = json.load(f)

print(f"Selected features : {len(selected)}")
print(f"AUC Test          : {results['auc_test']}")
print(f"AUC OOT           : {results['auc_oot']}")

Selected features : 27
AUC Test          : 0.9033
AUC OOT           : 0.9042


In [29]:
# load oot label
label_oot = pd.read_csv("D:/credit-card-crosssell/data/result_oot.csv")
y_proba = label_oot['propensity_score']
y_oot = label_oot['label']
# Load current data 
curr = pd.read_csv("D:/credit-card-crosssell/data/feature_store_current.csv")

drop_cols  = ['customer_id', 'observation_date']
X_curr     = curr.drop(columns=drop_cols)

# Encode 
X_curr_encoded = X_curr.copy()
for col in ['gender', 'province']:
    X_curr_encoded[col] = encoders[col].transform(X_curr[col])

# Score 
scores = best_model.predict_proba(X_curr_encoded[selected])[:, 1]

result = pd.DataFrame({
    'customer_id'     : curr['customer_id'],
    'observation_date': curr['observation_date'],
    'propensity_score': scores.round(4),
}).sort_values('propensity_score', ascending=False).reset_index(drop=True)

In [30]:
# Apply lên current data
y_curr_pred = (scores >= 0.24).astype(int)

print(f"\nKH predict = 1 (contact) : {y_curr_pred.sum():,}")
print(f"KH predict = 0 (skip)    : {(y_curr_pred == 0).sum():,}")


KH predict = 1 (contact) : 4,739
KH predict = 0 (skip)    : 261


In [31]:
# Apply threshold lên current data
y_curr_pred = (scores >= 0.24).astype(int)

curr_result = pd.DataFrame({
    'customer_id'     : curr['customer_id'],
    'observation_date': curr['observation_date'],
    'propensity_score': scores.round(4),
    'predicted_label' : y_curr_pred,
})

print(f"KH predict = 1 (contact) : {y_curr_pred.sum():,}")
print(f"KH predict = 0 (skip)    : {(y_curr_pred == 0).sum():,}")
print(f"\nScore distribution của predicted = 1:")
print(curr_result[curr_result['predicted_label']==1]['propensity_score'].describe())

KH predict = 1 (contact) : 4,739
KH predict = 0 (skip)    : 261

Score distribution của predicted = 1:
count    4739.000000
mean        0.744386
std         0.231421
min         0.240000
25%         0.529950
50%         0.816300
75%         0.972900
max         0.995600
Name: propensity_score, dtype: float64


# test design and sample size

test "campaign có hiệu quả không" và test cả "model có thực sự add value không".


### arms n purpose

| Arm | Predict | Contact | Mục đích |
|---|---|---|---|
| A | 1 | Yes | Model + campaign → ground truth của strategy |
| B | 1 | No | Model value thuần — không có campaign effect |
| C | 0/1 | No | Natural baseline — không làm gì |
| D | 0 | Yes | Random contact — campaign không có model |

**Test 1: A vs C** — Model + strategy có hiệu quả hơn không làm gì không?

```
H₀: p_A - p_C ≤ 0
→ Đây là overall business value của toàn bộ system
→ Câu hỏi quan trọng nhất với business
```

**Test 2: A vs B** — Contact có add value lên trên model không?

```
H₀: p_A - p_B ≤ 0
→ Isolate campaign uplift
→ Nếu A ≈ B → KH predicted = 1 sẽ tự convert dù không contact
→ Campaign waste money
```

**Test 3: B vs C** — Model predict = 1 có tự convert cao hơn không?

```
H₀: p_B - p_C ≤ 0
→ Đây là test model discrimination power
→ Nếu B > C → model đang identify đúng high-propensity KH
→ Quan trọng để validate model chứ không chỉ campaign
```

**Test 4: A vs D** — Model có tốt hơn random contact không?

```
H₀: p_A - p_D ≤ 0
→ Đây là business case của model
→ Nếu A ≈ D → model không add value, contact random cũng được
→ Nếu A > D → model worth the investment
```

**Test 5: D vs C** — Gọi bừa có hiệu quả hơn không làm gì không?

```
H₀: p_D - p_C ≤ 0
→ Test xem campaign channel itself có value không
→ Tách biệt channel effect khỏi model effect
```

## sample size

In [32]:
def sample_size(z_alpha, z_beta, p_bar, r, delta):
    """
    z_alpha : Z_{1- alpha} — từ mức ý nghĩa alpha (Type I error)
              alpha = 0.05 → z_alpha = 1.645 (one-sided)
    z_beta  : Z_{1-β} — từ power (1 - Type II error)
              power = 0.80 → z_beta = 0.842
    p_bar   : pooled conversion rate = (p_A + p_C) / 2
    r       : allocation ratio = n_A / n_C
              r = 2 → nhóm A gấp đôi nhóm C
    delta   : MDE = p_A - p_C
              delta = 0.05 → detect được effect ≥ 5%
    
    return  : n_C (cỡ mẫu nhóm nhỏ hơn)
              n_A = r * n_C
    """
    n = ((z_alpha + z_beta)**2 * p_bar * (1 - p_bar) * (r + 1)) / (r * delta**2)
    return int(np.ceil(n))

1. p_C baseline là bao nhiêu? → lấy từ historical data
2. MDE cho từng comparison là bao nhiêu?
3. Allocation ratio giữa 4 arms như thế nào?
   Equal (25/25/25/25) hay unequal?
4. Primary test là gì? → quyết định α

Chọn test strategy phân cấp priority:

```
Primary test   : A vs C (overall value)     → α = 0.05
Secondary tests: A vs B, B vs C, A vs D    → α = 0.025 (Bonferroni/2)
Exploratory    : D vs C                    → α = 0.10
```

In [33]:
# Primary test: A vs C
label_historical = pd.read_csv("D:/credit-card-crosssell/data/label_historical.csv")
p_C     = label_historical['converted'].mean()
p_A     = p_C + 0.05
p_bar   = (p_A + p_C) / 2
r       = 2           # n_A / n_C = 40% / 20%
delta   = p_A - p_C
alpha   = 0.05
power   = 0.80

z_alpha = stats.norm.ppf(1 - alpha)   # 1.645
z_beta  = stats.norm.ppf(power)       # 0.842

n_C = sample_size(z_alpha, z_beta, p_bar, r, delta)
n_A = r * n_C
n_B = n_C
n_D = n_C

total = n_A + n_B + n_C + n_D

In [34]:
print(f"z_alpha : {z_alpha:.3f}")
print(f"z_beta  : {z_beta:.3f}")
print(f"p_bar   : {p_bar:.4f}")
print(f"delta   : {delta:.4f}")
print(f"r       : {r}")
print()
print(f"n_C (control)          : {n_C:,}")
print(f"n_A (model + contact)  : {n_A:,}")
print(f"n_B (model, no contact): {n_B:,}")
print(f"n_D (random contact)   : {n_D:,}")
print(f"Total                  : {total:,}")
print(f"Current pool           : 5,000")
print(f"Sufficient             : {'Yes' if 5000 >= total else 'No'}")

z_alpha : 1.645
z_beta  : 0.842
p_bar   : 0.7361
delta   : 0.0500
r       : 2

n_C (control)          : 721
n_A (model + contact)  : 1,442
n_B (model, no contact): 721
n_D (random contact)   : 721
Total                  : 3,605
Current pool           : 5,000
Sufficient             : Yes


In [35]:
np.random.seed(42)

# Pool predict=1 → candidates cho Arm A và B
pool_1 = curr_result[curr_result['predicted_label'] == 1]['customer_id'].values
# Pool predict=0 → candidates cho Arm D
pool_0 = curr_result[curr_result['predicted_label'] == 0]['customer_id'].values

In [36]:
# Sample từng arm 

# Arm A: model + contact, predict=1, n=1442
arm_A = np.random.choice(pool_1, n_A, replace=False)

# Arm B: model no contact, predict=1, n=721
# Lấy từ pool_1 sau khi đã loại arm_A
remaining_1 = np.setdiff1d(pool_1, arm_A)
arm_B = np.random.choice(remaining_1, n_B, replace=False)

# Arm C: control, không contact, n=721
# Lấy từ toàn bộ pool (cả predict=0 và 1 chưa được assign)
remaining_all = np.setdiff1d(curr_result['customer_id'].values, 
                              np.concatenate([arm_A, arm_B]))
arm_C = np.random.choice(remaining_all, n_C, replace=False)

# Arm D: random contact, predict=0, n=721
remaining_0 = np.setdiff1d(pool_0, arm_C)
if len(remaining_0) >= n_D:
    arm_D = np.random.choice(remaining_0, n_D, replace=False)
else:
    # Nếu pool_0 không đủ thì lấy thêm từ remaining_all
    arm_D = np.random.choice(
        np.setdiff1d(remaining_all, arm_C), n_D, replace=False)

In [42]:
print(f"Arm A (model + contact)  : {len(arm_A):,}")
print(f"Arm B (model, no contact): {len(arm_B):,}")
print(f"Arm C (control)          : {len(arm_C):,}")
print(f"Arm D (random contact)   : {len(arm_D):,}")
print(f"Total                    : {len(arm_A)+len(arm_B)+len(arm_C)+len(arm_D):,}")

# Overlap check
all_ids = np.concatenate([arm_A, arm_B, arm_C, arm_D])
print(f"Unique KH                : {len(np.unique(all_ids)):,}")
print(f"Overlap                  : {'None' if len(np.unique(all_ids)) == len(all_ids) else 'Has overlap'}")

Arm A (model + contact)  : 1,442
Arm B (model, no contact): 721
Arm C (control)          : 721
Arm D (random contact)   : 721
Total                    : 3,605
Unique KH                : 3,605
Overlap                  : None


In [43]:
# export then claude gen label
# Tạo assignment table
assignment = pd.DataFrame({
    'customer_id': np.concatenate([arm_A, arm_B, arm_C, arm_D]),
    'arm'        : (['A'] * len(arm_A) + 
                    ['B'] * len(arm_B) + 
                    ['C'] * len(arm_C) + 
                    ['D'] * len(arm_D))
})

# Merge với propensity scores
assignment = assignment.merge(
    curr_result[['customer_id', 'propensity_score', 'predicted_label']],
    on='customer_id'
)

print(assignment.groupby('arm').agg(
    n                = ('customer_id', 'count'),
    avg_score        = ('propensity_score', 'mean'),
    pct_predicted_1  = ('predicted_label', 'mean')
).round(4))

# Save
assignment.to_csv(
    "D:/credit-card-crosssell/data/campaign_assignment.csv", 
    index=False
)
print("\nSaved: campaign_assignment.csv")

        n  avg_score  pct_predicted_1
arm                                  
A    1442     0.7505           1.0000
B     721     0.7531           1.0000
C     721     0.6781           0.9098
D     721     0.6897           0.9168

Saved: campaign_assignment.csv


## ab test

In [44]:
outcome = pd.read_csv("D:\credit-card-crosssell\data\campaign_outcomes.csv")

In [63]:
arm_A = outcome[outcome['arm'] == 'A']
arm_B = outcome[outcome['arm'] == 'B']
arm_C = outcome[outcome['arm'] == 'C']
arm_D = outcome[outcome['arm'] == 'D']

# Define n trước
n_A = len(arm_A)
n_B = len(arm_B)
n_C = len(arm_C)
n_D = len(arm_D)

conv_A = arm_A['converted'].sum()
conv_B = arm_B['converted'].sum()
conv_C = arm_C['converted'].sum()
conv_D = arm_D['converted'].sum()

# Rồi mới tính p
p_A = conv_A / n_A
p_B = conv_B / n_B
p_C = conv_C / n_C
p_D = conv_D / n_D

**Test 1: A vs C** — Model + strategy có hiệu quả hơn không làm gì không?

```
H₀: p_A - p_C ≤ 0
```

In [54]:
print(f"Arm A — model + contact")
print(f"  n         : {n_A:,}")
print(f"  converted : {conv_A:,}")
print(f"  rate      : {p_A:.4f}")
print()
print(f"Arm C — control")
print(f"  n         : {n_C:,}")
print(f"  converted : {conv_C:,}")
print(f"  rate      : {p_C:.4f}")
print()

# ── Test statistic ────────────────────────────────────────────
# H₀: p_A - p_C ≤ 0
# H₁: p_A - p_C > 0  (one-sided)
alpha = 0.05

p_bar  = (conv_A + conv_C) / (n_A + n_C)   # pooled
se     = np.sqrt(p_bar * (1 - p_bar) * (1/n_A + 1/n_C))
z_stat = (p_A - p_C) / se
p_val  = 1 - stats.norm.cdf(z_stat)        # one-sided

# ── Confidence interval ───────────────────────────────────────
z_crit   = stats.norm.ppf(1 - alpha)
ci_lower = (p_A - p_C) - z_crit * se
ci_upper = (p_A - p_C) + z_crit * se

# ── Results ───────────────────────────────────────────────────
print(f"── Test 1: A vs C (Primary) ──────────────────")
print(f"H₀: p_A - p_C ≤ 0")
print(f"H₁: p_A - p_C > 0  (one-sided, α = {alpha})")
print()
print(f"p̄ (pooled)  : {p_bar:.4f}")
print(f"SE           : {se:.4f}")
print(f"Δ observed   : {p_A - p_C:+.4f}")
print(f"Z statistic  : {z_stat:.4f}")
print(f"Z critical   : {z_crit:.4f}")
print(f"p-value      : {p_val:.4f}")
print(f"90% CI of Δ  : [{ci_lower:.4f}, {ci_upper:.4f}]")
print()
if p_val < alpha:
    print(f"Reject H₀ — campaign + model có hiệu quả hơn không làm gì")
    print(f"  Incremental conversion: {p_A - p_C:+.2%}")
else:
    print(f"Fail to reject H₀ — không đủ evidence")

Arm A — model + contact
  n         : 1,442
  converted : 1,148
  rate      : 0.7961

Arm C — control
  n         : 721
  converted : 526
  rate      : 0.7295

── Test 1: A vs C (Primary) ──────────────────
H₀: p_A - p_C ≤ 0
H₁: p_A - p_C > 0  (one-sided, α = 0.05)

p̄ (pooled)  : 0.7739
SE           : 0.0191
Δ observed   : +0.0666
Z statistic  : 3.4894
Z critical   : 1.6449
p-value      : 0.0002
90% CI of Δ  : [0.0352, 0.0980]

Reject H₀ — campaign + model có hiệu quả hơn không làm gì
  Incremental conversion: +6.66%



**Test 2: A vs B** — Contact có add value lên trên model không?

```
H₀: p_A - p_B ≤ 0
```


In [57]:
# ── Test 2: A vs B — Secondary ────────────────────────────────
print(f"Arm A — model + contact")
print(f"  n         : {n_A:,}")
print(f"  converted : {conv_A:,}")
print(f"  rate      : {p_A:.4f}")
print()
print(f"Arm B — model, no contact")
print(f"  n         : {n_B:,}")
print(f"  converted : {conv_B:,}")
print(f"  rate      : {p_B:.4f}")
print()

alpha  = 0.025
p_bar  = (conv_A + conv_B) / (n_A + n_B)
se     = np.sqrt(p_bar * (1 - p_bar) * (1/n_A + 1/n_B))
z_stat = (p_A - p_B) / se
p_val  = 1 - stats.norm.cdf(z_stat)

z_crit   = stats.norm.ppf(1 - alpha)
ci_lower = (p_A - p_B) - z_crit * se
ci_upper = (p_A - p_B) + z_crit * se

print(f"── Test 2: A vs B (Secondary) ──────────────────")
print(f"H₀: p_A - p_B ≤ 0")
print(f"H₁: p_A - p_B > 0  (one-sided, α = {alpha})")
print()
print(f"p̄ (pooled)   : {p_bar:.4f}")
print(f"SE            : {se:.4f}")
print(f"Δ observed    : {p_A - p_B:+.4f}")
print(f"Z statistic   : {z_stat:.4f}")
print(f"Z critical    : {z_crit:.4f}")
print(f"p-value       : {p_val:.4f}")
print(f"97.5% CI of Δ : [{ci_lower:.4f}, {ci_upper:.4f}]")
print()
if p_val < alpha:
    print(f"Reject H₀ — contact add value lên trên model")
    print(f"  Incremental conversion: {p_A - p_B:+.2%}")
else:
    print(f"Fail to reject H₀ — không đủ evidence")

Arm A — model + contact
  n         : 1,442
  converted : 1,148
  rate      : 0.7961

Arm B — model, no contact
  n         : 721
  converted : 547
  rate      : 0.7587

── Test 2: A vs B (Secondary) ──────────────────
H₀: p_A - p_B ≤ 0
H₁: p_A - p_B > 0  (one-sided, α = 0.025)

p̄ (pooled)   : 0.7836
SE            : 0.0188
Δ observed    : +0.0374
Z statistic   : 1.9939
Z critical    : 1.9600
p-value       : 0.0231
97.5% CI of Δ : [0.0006, 0.0743]

Reject H₀ — contact add value lên trên model
  Incremental conversion: +3.74%



**Test 3: B vs C** — Model predict = 1 có tự convert cao hơn không?

```
H₀: p_B - p_C ≤ 0
```

In [58]:
# Test 3: B vs C — Secondary 
print(f"Arm B — model, no contact")
print(f"  n         : {n_B:,}")
print(f"  converted : {conv_B:,}")
print(f"  rate      : {p_B:.4f}")
print()
print(f"Arm C — control")
print(f"  n         : {n_C:,}")
print(f"  converted : {conv_C:,}")
print(f"  rate      : {p_C:.4f}")
print()

alpha  = 0.025
p_bar  = (conv_B + conv_C) / (n_B + n_C)
se     = np.sqrt(p_bar * (1 - p_bar) * (1/n_B + 1/n_C))
z_stat = (p_B - p_C) / se
p_val  = 1 - stats.norm.cdf(z_stat)

z_crit   = stats.norm.ppf(1 - alpha)
ci_lower = (p_B - p_C) - z_crit * se
ci_upper = (p_B - p_C) + z_crit * se

print(f"Test 3: B vs C (Secondary)")
print(f"H₀: p_B - p_C ≤ 0")
print(f"H₁: p_B - p_C > 0  (one-sided, α = {alpha})")
print()
print(f"p̄ (pooled)   : {p_bar:.4f}")
print(f"SE            : {se:.4f}")
print(f"Δ observed    : {p_B - p_C:+.4f}")
print(f"Z statistic   : {z_stat:.4f}")
print(f"Z critical    : {z_crit:.4f}")
print(f"p-value       : {p_val:.4f}")
print(f"97.5% CI of Δ : [{ci_lower:.4f}, {ci_upper:.4f}]")
print()
if p_val < alpha:
    print(f"✓ Reject H₀ — model identify đúng high-propensity KH")
    print(f"  Incremental conversion: {p_B - p_C:+.2%}")
else:
    print(f"✗ Fail to reject H₀ — không đủ evidence")

Arm B — model, no contact
  n         : 721
  converted : 547
  rate      : 0.7587

Arm C — control
  n         : 721
  converted : 526
  rate      : 0.7295

Test 3: B vs C (Secondary)
H₀: p_B - p_C ≤ 0
H₁: p_B - p_C > 0  (one-sided, α = 0.025)

p̄ (pooled)   : 0.7441
SE            : 0.0230
Δ observed    : +0.0291
Z statistic   : 1.2673
Z critical    : 1.9600
p-value       : 0.1025
97.5% CI of Δ : [-0.0159, 0.0742]

✗ Fail to reject H₀ — không đủ evidence



**Test 4: A vs D** — Model có tốt hơn random contact không?

```
H₀: p_A - p_D ≤ 0
```

In [61]:
# Test 4: A vs D — Secondary 
print(f"Arm A — model + contact")
print(f"  n         : {n_A:,}")
print(f"  converted : {conv_A:,}")
print(f"  rate      : {p_A:.4f}")
print()
print(f"Arm D — random contact")
print(f"  n         : {n_D:,}")
print(f"  converted : {conv_D:,}")
print(f"  rate      : {p_D:.4f}")
print()

alpha  = 0.025
p_bar  = (conv_A + conv_D) / (n_A + n_D)
se     = np.sqrt(p_bar * (1 - p_bar) * (1/n_A + 1/n_D))
z_stat = (p_A - p_D) / se
p_val  = 1 - stats.norm.cdf(z_stat)

z_crit   = stats.norm.ppf(1 - alpha)
ci_lower = (p_A - p_D) - z_crit * se
ci_upper = (p_A - p_D) + z_crit * se

print(f" Test 4: A vs D (Secondary) ")
print(f"H₀: p_A - p_D ≤ 0")
print(f"H₁: p_A - p_D > 0  (one-sided, α = {alpha})")
print()
print(f"p̄ (pooled)   : {p_bar:.4f}")
print(f"SE            : {se:.4f}")
print(f"Δ observed    : {p_A - p_D:+.4f}")
print(f"Z statistic   : {z_stat:.4f}")
print(f"Z critical    : {z_crit:.4f}")
print(f"p-value       : {p_val:.4f}")
print(f"97.5% CI of Δ : [{ci_lower:.4f}, {ci_upper:.4f}]")
print()
if p_val < alpha:
    print(f"Reject H₀ — model tốt hơn random contact")
    print(f"  Incremental conversion: {p_A - p_D:+.2%}")
else:
    print(f"Fail to reject H₀ — không đủ evidence")

Arm A — model + contact
  n         : 1,442
  converted : 1,148
  rate      : 0.7961

Arm D — random contact
  n         : 721
  converted : 509
  rate      : 0.7060

 Test 4: A vs D (Secondary) 
H₀: p_A - p_D ≤ 0
H₁: p_A - p_D > 0  (one-sided, α = 0.025)

p̄ (pooled)   : 0.7661
SE            : 0.0193
Δ observed    : +0.0902
Z statistic   : 4.6690
Z critical    : 1.9600
p-value       : 0.0000
97.5% CI of Δ : [0.0523, 0.1280]

Reject H₀ — model tốt hơn random contact
  Incremental conversion: +9.02%



**Test 5: D vs C** — Gọi bừa có hiệu quả hơn không làm gì không?

```
H₀: p_D - p_C ≤ 0
```

In [62]:
# Test 5: D vs C — Exploratory 
print(f"Arm D — random contact")
print(f"  n         : {n_D:,}")
print(f"  converted : {conv_D:,}")
print(f"  rate      : {p_D:.4f}")
print()
print(f"Arm C — control")
print(f"  n         : {n_C:,}")
print(f"  converted : {conv_C:,}")
print(f"  rate      : {p_C:.4f}")
print()

alpha  = 0.10
p_bar  = (conv_D + conv_C) / (n_D + n_C)
se     = np.sqrt(p_bar * (1 - p_bar) * (1/n_D + 1/n_C))
z_stat = (p_D - p_C) / se
p_val  = 1 - stats.norm.cdf(z_stat)

z_crit   = stats.norm.ppf(1 - alpha)
ci_lower = (p_D - p_C) - z_crit * se
ci_upper = (p_D - p_C) + z_crit * se

print(f" Test 5: D vs C (Exploratory)")
print(f"H₀: p_D - p_C ≤ 0")
print(f"H₁: p_D - p_C > 0  (one-sided, α = {alpha})")
print()
print(f"p̄ (pooled)   : {p_bar:.4f}")
print(f"SE            : {se:.4f}")
print(f"Δ observed    : {p_D - p_C:+.4f}")
print(f"Z statistic   : {z_stat:.4f}")
print(f"Z critical    : {z_crit:.4f}")
print(f"p-value       : {p_val:.4f}")
print(f"90% CI of Δ   : [{ci_lower:.4f}, {ci_upper:.4f}]")
print()
if p_val < alpha:
    print(f"✓ Reject H₀ — random contact có hiệu quả")
    print(f"  Incremental conversion: {p_D - p_C:+.2%}")
else:
    print(f"✗ Fail to reject H₀ — random contact không hiệu quả")
    print(f"  Δ = {p_D - p_C:+.2%} — Do Not Disturb effect có thể xảy ra")

Arm D — random contact
  n         : 721
  converted : 509
  rate      : 0.7060

Arm C — control
  n         : 721
  converted : 526
  rate      : 0.7295

 Test 5: D vs C (Exploratory)
H₀: p_D - p_C ≤ 0
H₁: p_D - p_C > 0  (one-sided, α = 0.1)

p̄ (pooled)   : 0.7178
SE            : 0.0237
Δ observed    : -0.0236
Z statistic   : -0.9946
Z critical    : 1.2816
p-value       : 0.8400
90% CI of Δ   : [-0.0540, 0.0068]

✗ Fail to reject H₀ — random contact không hiệu quả
  Δ = -2.36% — Do Not Disturb effect có thể xảy ra


In [68]:
cost_per_contact = 83_000
revenue_per_conv = 15_000_000

# Absolute ROI từng arm
# Revenue = số KH convert × revenue mỗi KH
# Cost    = số KH được contact × cost mỗi contact

revenue_A = conv_A * revenue_per_conv
cost_A    = n_A * cost_per_contact      # toàn bộ arm A được contact
net_A     = revenue_A - cost_A

revenue_B = conv_B * revenue_per_conv
cost_B    = 0                           # arm B không được contact
net_B     = revenue_B - cost_B

revenue_C = conv_C * revenue_per_conv
cost_C    = 0                           # control không được contact
net_C     = revenue_C - cost_C

revenue_D = conv_D * revenue_per_conv
cost_D    = n_D * cost_per_contact      # arm D được contact
net_D     = revenue_D - cost_D

# Incremental ROI vs control 
# Incremental revenue = (actual conv - expected conv nếu không làm gì) × revenue
# Expected conv nếu không làm gì = p_C × n_arm

inc_revenue_A = (conv_A - p_C * n_A) * revenue_per_conv
inc_cost_A    = n_A * cost_per_contact
inc_roi_A     = inc_revenue_A - inc_cost_A

inc_revenue_B = (conv_B - p_C * n_B) * revenue_per_conv
inc_cost_B    = 0
inc_roi_B     = inc_revenue_B - inc_cost_B

inc_revenue_D = (conv_D - p_C * n_D) * revenue_per_conv
inc_cost_D    = n_D * cost_per_contact
inc_roi_D     = inc_revenue_D - inc_cost_D

# Per KH metrics 
roi_per_kh_A  = net_A / n_A
roi_per_kh_B  = net_B / n_B
roi_per_kh_C  = net_C / n_C
roi_per_kh_D  = net_D / n_D

In [69]:
# Print 
print("=== Absolute ROI ===")
print(f"{'Arm':<6} {'Revenue':>18} {'Cost':>15} {'Net ROI':>18} {'ROI/KH':>15}")
print("-" * 75)
for arm, rev, cost, net, n in [
    ('A', revenue_A, cost_A, net_A, n_A),
    ('B', revenue_B, cost_B, net_B, n_B),
    ('C', revenue_C, cost_C, net_C, n_C),
    ('D', revenue_D, cost_D, net_D, n_D),
]:
    print(f"{arm:<6} {rev:>18,.0f} {cost:>15,.0f} {net:>18,.0f} {net/n:>15,.0f}")

print()
print("=== Incremental ROI vs Control (C) ===")
print(f"{'Arm':<6} {'Inc. Revenue':>18} {'Inc. Cost':>15} {'Inc. ROI':>18}")
print("-" * 60)
for arm, inc_rev, inc_cost, inc_roi in [
    ('A', inc_revenue_A, inc_cost_A, inc_roi_A),
    ('B', inc_revenue_B, inc_cost_B, inc_roi_B),
    ('D', inc_revenue_D, inc_cost_D, inc_roi_D),
]:
    print(f"{arm:<6} {inc_rev:>18,.0f} {inc_cost:>15,.0f} {inc_roi:>18,.0f}")

print()
print("=== Business Conclusion ===")
print(f"Best strategy         : Arm A (model + contact)")
print(f"Incremental ROI of A  : {inc_roi_A:,.0f} VND")
print(f"Cost of model (B vs D): {inc_roi_B - inc_roi_D:,.0f} VND incremental value")
print(f"Random contact ROI    : {inc_roi_D:,.0f} VND — {'waste' if inc_roi_D < 0 else 'positive'}")

print("=== ROMI ===")
print(f"Arm A: {inc_roi_A / cost_A:.2f}x  → mỗi 1 VND bỏ ra thu về {inc_roi_A/cost_A:.2f} VND incremental")
print(f"Arm D: {inc_roi_D / cost_D:.2f}x  → mỗi 1 VND bỏ ra thu về {inc_roi_D/cost_D:.2f} VND incremental")

=== Absolute ROI ===
Arm               Revenue            Cost            Net ROI          ROI/KH
---------------------------------------------------------------------------
A          17,220,000,000     119,686,000     17,100,314,000      11,858,748
B           8,205,000,000               0      8,205,000,000      11,380,028
C           7,890,000,000               0      7,890,000,000      10,943,135
D           7,635,000,000      59,843,000      7,575,157,000      10,506,459

=== Incremental ROI vs Control (C) ===
Arm          Inc. Revenue       Inc. Cost           Inc. ROI
------------------------------------------------------------
A           1,440,000,000     119,686,000      1,320,314,000
B             315,000,000               0        315,000,000
D            -255,000,000      59,843,000       -314,843,000

=== Business Conclusion ===
Best strategy         : Arm A (model + contact)
Incremental ROI of A  : 1,320,314,000 VND
Cost of model (B vs D): 629,843,000 VND incremental va

# ROI Analysis — Credit Card Cross-sell Campaign

In [71]:
from IPython.display import display, Markdown

display(Markdown(f"""
# ROI Analysis — Credit Card Cross-sell Campaign

## Setup

| Parameter | Value |
|---|---|
| Cost per contact | 83,000 VND |
| Revenue per conversion | 15,000,000 VND |
| Label window | 1/7/2025 – 30/9/2025 |

---

## Arm Summary

| Arm | Description | N | Converted | Conv. Rate |
|---|---|---|---|---|
| A | Model + contact | {n_A:,} | {conv_A:,} | {p_A:.2%} |
| B | Model, no contact | {n_B:,} | {conv_B:,} | {p_B:.2%} |
| C | Control (baseline) | {n_C:,} | {conv_C:,} | {p_C:.2%} |
| D | Random contact | {n_D:,} | {conv_D:,} | {p_D:.2%} |

---

## Absolute ROI

| Arm | Revenue | Cost | Net ROI | ROI/KH |
|---|---|---|---|---|
| A | {revenue_A:,.0f} | {cost_A:,.0f} | {net_A:,.0f} | {net_A/n_A:,.0f} |
| B | {revenue_B:,.0f} | {cost_B:,.0f} | {net_B:,.0f} | {net_B/n_B:,.0f} |
| C | {revenue_C:,.0f} | {cost_C:,.0f} | {net_C:,.0f} | {net_C/n_C:,.0f} |
| D | {revenue_D:,.0f} | {cost_D:,.0f} | {net_D:,.0f} | {net_D/n_D:,.0f} |

---

## Incremental ROI vs Control (C)

> Incremental Revenue = (Actual conversions − Expected conversions at p_C) × Revenue/conv  
> Expected conversions at p_C = {p_C:.2%} × n_arm

| Arm | Inc. Revenue | Inc. Cost | Inc. ROI |
|---|---|---|---|
| A | {inc_revenue_A:,.0f} | {inc_cost_A:,.0f} | {inc_roi_A:,.0f} |
| B | {inc_revenue_B:,.0f} | {inc_cost_B:,.0f} | {inc_roi_B:,.0f} |
| D | {inc_revenue_D:,.0f} | {inc_cost_D:,.0f} | {inc_roi_D:,.0f} |

---

## ROMI (Return on Marketing Investment)

| Arm | ROMI | Interpretation |
|---|---|---|
| A | {inc_roi_A/cost_A:.2f}x | Mỗi 1 VND bỏ ra thu về {inc_roi_A/cost_A:.2f} VND incremental |
| D | {inc_roi_D/cost_D:.2f}x | Mỗi 1 VND bỏ ra thu về {inc_roi_D/cost_D:.2f} VND incremental |

---

## Key Insights

**1. Model + Contact (A) là strategy tốt nhất**
- Incremental ROI: {inc_roi_A:,.0f} VND (~{inc_roi_A/1e9:.2f} tỷ)
- ROMI: {inc_roi_A/cost_A:.2f}x — mỗi đồng bỏ ra thu về {inc_roi_A/cost_A:.2f} đồng

**2. Model alone (B) vẫn có value**
- Incremental ROI: {inc_roi_B:,.0f} VND (~{inc_roi_B/1e9:.2f} tỷ)
- Chỉ bằng cách identify đúng người, không cần tốn chi phí contact

**3. Random contact (D) destroy value**
- Incremental ROI: {inc_roi_D:,.0f} VND
- ROMI: {inc_roi_D/cost_D:.2f}x — contact sai người không chỉ waste money mà còn gây hại

**4. Value của model vs random contact**
- B − D = {inc_roi_B - inc_roi_D:,.0f} VND
- Đây là incremental value mà model tạo ra so với không dùng model

---

## Business Recommendation

- Deploy **Arm A strategy** — chỉ contact KH được model identify (predict = 1)
- Không contact random → tránh destroy value
- Scale campaign ra {(y_curr_pred == 1).sum():,} KH predict = 1 trong pool hiện tại
- Expected incremental revenue ≈ {inc_roi_A/n_A * (y_curr_pred == 1).sum():,.0f} VND nếu scale toàn bộ
"""))


# ROI Analysis — Credit Card Cross-sell Campaign

## Setup

| Parameter | Value |
|---|---|
| Cost per contact | 83,000 VND |
| Revenue per conversion | 15,000,000 VND |
| Label window | 1/7/2025 – 30/9/2025 |

---

## Arm Summary

| Arm | Description | N | Converted | Conv. Rate |
|---|---|---|---|---|
| A | Model + contact | 1,442 | 1,148 | 79.61% |
| B | Model, no contact | 721 | 547 | 75.87% |
| C | Control (baseline) | 721 | 526 | 72.95% |
| D | Random contact | 721 | 509 | 70.60% |

---

## Absolute ROI

| Arm | Revenue | Cost | Net ROI | ROI/KH |
|---|---|---|---|---|
| A | 17,220,000,000 | 119,686,000 | 17,100,314,000 | 11,858,748 |
| B | 8,205,000,000 | 0 | 8,205,000,000 | 11,380,028 |
| C | 7,890,000,000 | 0 | 7,890,000,000 | 10,943,135 |
| D | 7,635,000,000 | 59,843,000 | 7,575,157,000 | 10,506,459 |

---

## Incremental ROI vs Control (C)

> Incremental Revenue = (Actual conversions − Expected conversions at p_C) × Revenue/conv  
> Expected conversions at p_C = 72.95% × n_arm

| Arm | Inc. Revenue | Inc. Cost | Inc. ROI |
|---|---|---|---|
| A | 1,440,000,000 | 119,686,000 | 1,320,314,000 |
| B | 315,000,000 | 0 | 315,000,000 |
| D | -255,000,000 | 59,843,000 | -314,843,000 |

---

## ROMI (Return on Marketing Investment)

| Arm | ROMI | Interpretation |
|---|---|---|
| A | 11.03x | Mỗi 1 VND bỏ ra thu về 11.03 VND incremental |
| D | -5.26x | Mỗi 1 VND bỏ ra thu về -5.26 VND incremental |

---

## Key Insights

**1. Model + Contact (A) là strategy tốt nhất**
- Incremental ROI: 1,320,314,000 VND (~1.32 tỷ)
- ROMI: 11.03x — mỗi đồng bỏ ra thu về 11.03 đồng

**2. Model alone (B) vẫn có value**
- Incremental ROI: 315,000,000 VND (~0.32 tỷ)
- Chỉ bằng cách identify đúng người, không cần tốn chi phí contact

**3. Random contact (D) destroy value**
- Incremental ROI: -314,843,000 VND
- ROMI: -5.26x — contact sai người không chỉ waste money mà còn gây hại

**4. Value của model vs random contact**
- B − D = 629,843,000 VND
- Đây là incremental value mà model tạo ra so với không dùng model

---

## Business Recommendation

- Deploy **Arm A strategy** — chỉ contact KH được model identify (predict = 1)
- Không contact random → tránh destroy value
- Scale campaign ra 4,739 KH predict = 1 trong pool hiện tại
- Expected incremental revenue ≈ 4,339,090,184 VND nếu scale toàn bộ
